In [1]:
import geopandas as gpd
import rasterio
import rasterio.mask
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

In [2]:
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"

world = gpd.read_file(url)

In [3]:
germany = world[world["ADMIN"] == "Germany"]

In [4]:
src = rasterio.open("JRC-ESTAT_Census_Population_2021_100m.tif")

germany = germany.to_crs(src.crs)

In [5]:
out_image, out_transform = rasterio.mask.mask(
    src,
    germany.geometry,
    crop=True
)

In [6]:
rows, cols = np.where(out_image[0] > 0)

values = out_image[0][rows, cols]

xs, ys = rasterio.transform.xy(out_transform, rows, cols)

In [7]:
df = pd.DataFrame({
    "x": xs,
    "y": ys,
    "population": values
})

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

df["geometry"] = df.apply(lambda r: Point(r["x"], r["y"]), axis=1)

gdf = gpd.GeoDataFrame(df, geometry="geometry", crs="EPSG:3035")

gdf = gdf.to_crs(epsg=4326)

In [24]:
import geopandas as gpd
from shapely.geometry import Point

# 1. Geometrie korrekt (aus 3035)
gdf = gpd.GeoDataFrame(
    df.copy(),
    geometry=gpd.points_from_xy(df["x"], df["y"]),
    crs="EPSG:3035"
)

# 2. Zielpunkt (WGS84!)
target = Point(13.749733, 50.734608)

# 3. In Meter-Projektion für Distanz
gdf_m = gdf.to_crs(3857)
target_m = gpd.GeoSeries([target], crs=4326).to_crs(3857).iloc[0]

# 4. Distanz berechnen
gdf_m["dist_m"] = gdf_m.geometry.distance(target_m)

# 5. Nächsten Punkt holen
nearest = gdf_m.nsmallest(1, "dist_m")

nearest

,x,y,population,geometry,dist_m
2490046,4580550.0,3086150.0,3,POINT (1523462.994 6590931.86),17936.186506


In [21]:
print(nearest)

                 x          y  population                        geometry  \
2490046  4580550.0  3086150.0           3  POINT (1523462.994 6590931.86)   

                 dist  
2490046  17936.186506  


In [9]:
gdf.to_csv("../processed/density.csv",
                index=False,
                encoding="utf-8")